The goal of this notebook is to compare the region predictions for L2 and L3 cell types to the region predictions of their best matches from other atlases.

For that matter the top n predicted regions per cell type and match will computed and the overlap analyzed.

To run the notebook you will need an anndata object of [HypoMap (2022)](https://www.nature.com/articles/s42255-022-00657-y#data-availability). 

In [1]:
import os 
from pathlib import Path

from dotenv import dotenv_values
import anndata as ad
import pandas as pd
import numpy as np

In [2]:
# Setting up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

OUTDIR = DATADIR / "10_analyses" / "region_comparison"
os.makedirs(OUTDIR, exist_ok=True)

RM_FP = DATADIR / "9_objects" / "spatial" / "regionwise_relative_masses" / 'regionwise_relative_masses.csv'

TRANSLATOR_FP = DATADIR / "10_analyses" / "clustering_comparison" / 'HypoMap3D_to_other_atlas_celltype_mapper.csv'
REGION_META_FP = DATADIR / "7_spatial" / "abc_region_metadata.csv"

HMV1_FP = "/cephfs/scratch/bruening_scratch/lsteuernagel/data/hypoMap_publication/hypoMap_anndata.h5ad"
ZHANG_FP = DATADIR / "7_spatial" / "bounded" / "SP2-02_no_qc.h5ad"
YAO_FP = DATADIR / "7_spatial" / "bounded" / "SP2-01_no_qc.h5ad"

YAO_W_ABC_OBS_FP = DATADIR / "7_spatial" / "raw" / "Yao_spatial.h5ad"
ZHANG_W_ABC_OBS_FP = DATADIR / "7_spatial" / "raw" / "Zhang_spatial.h5ad"

# Let's load the necessary data

In [3]:
# cell type mapper
translator = pd.read_csv(TRANSLATOR_FP) 

# region wise relative masses
hm3d = pd.read_csv(RM_FP)

# HMv1 
hmv1 = ad.read_h5ad(HMV1_FP, backed="r")

# ABC ST datasets 
zhang = ad.read_h5ad(ZHANG_FP, backed="r")
yao = ad.read_h5ad(YAO_FP, backed="r")

/REDACTED-path/HypoMap3D/.venv/lib/python3.11/site-packages/anndata/_io/h5ad.py:192: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  adata = AnnData(**d)


In [4]:
# load subclass and supertype metadata from raw objects 
zhang_cells = [c.split("_")[-1] for c in list(zhang.obs_names)]
yao_cells = [c.split("_")[-1] for c in list(yao.obs_names)]

zhang_meta = ad.read_h5ad(ZHANG_W_ABC_OBS_FP, backed="r").obs.loc[zhang_cells, ["subclass", "supertype"]]
yao_meta = ad.read_h5ad(YAO_W_ABC_OBS_FP, backed="r").obs.loc[yao_cells, ["subclass", "supertype"]]

In [5]:
# add ABC clusters to .obs 
zhang.obs[["subclass", "supertype"]] = zhang_meta.values
yao.obs[["subclass", "supertype"]] = yao_meta.values

# rename columns as to match other objects
zhang.obs = zhang.obs.rename(columns={"RegionAcronym": "allen_acronym"})
yao.obs = yao.obs.rename(columns={"RegionAcronym": "allen_acronym"})

# add ccf to obs for integration into old code base 
zhang.obs[["x_ccf", "y_ccf", "z_ccf"]] = zhang.obsm["coords_ccf_mm"].loc[:, ["x", "y", "z"]]
yao.obs[["x_ccf", "y_ccf", "z_ccf"]] = yao.obsm["coords_ccf_mm"].loc[:, ["x", "y", "z"]]

In [6]:
# ABC region metadata
region_meta = pd.read_csv(REGION_META_FP)
name_to_acro_mapper = region_meta.set_index("name").to_dict()["acronym"]

# Let's generate/extract the predcitions

You need to run all below cells once for L2 and L3.

In [41]:
hm3d_level_to_analyze = "L3" # user input 

if hm3d_level_to_analyze == "L2":
    allen_level_to_analyze = "subclass"
    hmv1_level_to_analyze = "C66_named"
    
elif hm3d_level_to_analyze == "L3":
    allen_level_to_analyze = "supertype"
    hmv1_level_to_analyze = "C286_named"


HypoMapv1 has some nuclei summed up into one structure. For comparability, we to sum up the relative masses for HM3D and ABC rather than splitting the combined HMv1 structure into its substructure which would require the assumption of equal relative masses in all the subnuclei.

In [42]:
hmv1_like_acro_summarizer = {
    "MMp": "MM",
    "MMm": "MM",
    "MMl": "MM",
    "MMme": "MM", 
    "MMd": "MM",

    "TMd": "TM",
    "TMv": "TM"
}

## HypoMap3D

In [43]:
hm3d_pred = hm3d.loc[hm3d.celltype_id.str[2] == hm3d_level_to_analyze[-1], :]
hm3d_pred = hm3d_pred.drop(columns=["celltype_id", "celltype_abc_like", "flag"])
hm3d_pred = hm3d_pred.set_index("celltype_descr")
hm3d_pred = hm3d_pred.T
hm3d_pred = hm3d_pred.reset_index(names="allen_acronym")
hm3d_pred.allen_acronym = hm3d_pred.allen_acronym.replace(hmv1_like_acro_summarizer)
hm3d_pred = hm3d_pred.set_index("allen_acronym")
hm3d_pred = hm3d_pred.groupby("allen_acronym").sum()

In [44]:
hm3d_pred.head(3)

celltype_descr,Gb3-124 MPO Meis1,Gb3-125 SI Lgr5,Gb3-126 MPO Sema3d,Gb3-127 MPO Penk,Gb3-128 SI Chat,Gb3-129 NDB Fign,Gb3-130 ZI Hmcn1,Gb3-131 LHA Arx,Gb3-132 CP Nfix,Gb3-133 ZI Onecut3,...,NN3-26 Tanycyte_alpha2,NN3-27 Tanycyte_alpha1,NN3-28 Ependymal_1,NN3-3 Oligo_2_2,NN3-4 Oligo_2_3,NN3-5 OPC_1,NN3-6 Microglia_1_1,NN3-7 Microglia_2_1,NN3-8 Microglia_2_2,NN3-9 BAM_1
allen_acronym,,,,,,,,,,,,,,,,,,,,,
AAA,0.005764,0.036498,0.019285,0.004348,0.020334,2.234217e-02,0.007421,0.019952,0.011551,0.001176,...,0.002051,0.000959,0.000358,0.001074,0.022045,0.008278,0.007816,0.006200,9.737101e-03,0.012559
ACB,0.010683,0.020769,0.019905,0.007399,0.026565,7.528130e-03,0.007363,0.007376,0.022341,0.000232,...,0.010410,0.002228,0.002657,0.007512,0.041020,0.016412,0.016472,0.013210,1.899452e-02,0.015046
AD,0.000012,0.000006,0.000002,0.000115,0.000032,1.106238e-07,0.000000,0.000000,0.000054,0.000020,...,0.000019,0.000005,0.000013,0.000055,0.000002,0.000060,0.000046,0.000041,3.614046e-07,0.000017


## HypoMap v1

In [45]:
hmv1_pred = hmv1.obs.loc[:, ["Region_predicted", hmv1_level_to_analyze]]
hmv1_pred["acronym"] = hmv1_pred.Region_predicted.map(name_to_acro_mapper)
hmv1_pred.loc[hmv1_pred.Region_predicted == "Tuberomamillary nucleus", "acronym"] = "TM" # add TM  
hmv1_pred.loc[hmv1_pred.Region_predicted == "NA", "acronym"] = "NA"

In [46]:
hmv1_pred[["Region_predicted", "acronym"]].drop_duplicates()

,Region_predicted,acronym
AAACCCAAGGGTGAAA-1_SRR14252354,Paraventricular hypothalamic nucleus,PVH
AAACCCACAGCACACC-1_SRR14252354,NA,NA
AAACCCATCCCGAAAT-1_SRR14252354,Medial mammillary nucleus,MM
AAACGAAAGACCAGCA-1_SRR14252354,"Periventricular hypothalamic nucleus, posterio...",PVp
AAACGAAAGGTCCAGA-1_SRR14252354,Zona incerta,ZI
AAACGAAAGTGTAGTA-1_SRR14252354,Lateral hypothalamic area,LHA
AAACGAATCGCTGACG-1_SRR14252354,Ventromedial hypothalamic nucleus,VMH
AAACGCTGTATGAGAT-1_SRR14252354,Arcuate hypothalamic nucleus,ARH
AAAGAACTCAAACGTC-1_SRR14252354,Supramammillary nucleus,SUM
AAAGGATAGGGTTTCT-1_SRR14252354,Anterior hypothalamic nucleus,AHN


In [47]:
hmv1_pred = hmv1_pred.dropna()

hmv1_pred = (
    hmv1_pred
    .groupby(["acronym", hmv1_level_to_analyze])
    .size()
    .unstack(fill_value=0)
)

hmv1_pred = hmv1_pred.div(hmv1_pred.sum(axis=0), axis=1)

add nuclei that were not predicted

In [48]:
# add back all nuclei that were not predicted

not_included_nucs = [n for n in hm3d_pred.index if n not in hmv1_pred.index]

zeros_df = pd.DataFrame(
    index = not_included_nucs,
    columns = hmv1_pred.columns,
    data = np.zeros(shape=(len(not_included_nucs), len(hmv1_pred.columns)))
)

hmv1_pred = pd.concat([hmv1_pred, zeros_df], axis=0)

#reorder index. this will drop NA "nucleus"
hmv1_pred = hmv1_pred.loc[hm3d_pred.index, :] 

In [49]:
hmv1_pred.head(2)

C286_named,C286-100: Cxcl14.Pnoc.Mixed.GABA-2,C286-101: Zic1.Pnoc.Mixed.GABA-2,C286-102: Unassigned.C1ql2.Mixed.GABA-2,C286-103: Nfix.Unassigned.Mixed.GABA-2,C286-104: Pvalb.Mixed.GABA-2,C286-105: Otx2.Mixed.GABA-2,C286-106: Vip.Vipr2.GABA-2,C286-107: Cd79a.Fam122b.Vipr2.GABA-2,C286-108: Ppp1r17.Fam122b.Vipr2.GABA-2,C286-109: Chodl.Fam122b.Vipr2.GABA-2,...,C286-91: Prkch.GLU-8,C286-92: Gnrh1.GLU-9,C286-93: Csta2.Thrsp.Mixed.GABA-2,C286-94: Nfix.Thrsp.Mixed.GABA-2,C286-95: Meis2.Mixed.GABA-2,C286-96: Unassigned.Unassigned.Mixed.GABA-2,C286-97: Zic3.Unassigned.Mixed.GABA-2,C286-98: Pou6f2.Unassigned.Mixed.GABA-2,C286-99: Unassigned.Pnoc.Mixed.GABA-2,C286-9: Nfia.Nxph4.GLU-2
allen_acronym,,,,,,,,,,,,,,,,,,,,,
AAA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACB,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## ABC

For ABC, there is no supplemental data in their publications that would have cluster: all regions mappings. 

There is a [file](https://static-content.springer.com/esm/art%3A10.1038%2Fs41586-023-06812-z/MediaObjects/41586_2023_6812_MOESM8_ESM.xlsx) in the Yao publication where ABC clusters are mapped to CF_acronym.freq (with that column annotated as "Distribution of finer CCF structures based on CCF registration. Formatted as CCF structures followed by the fraction of mapped MERFISH cells registered to the given structure, sorted from highest to lowest till 80% cells are covered"). However, this is per cluster, so you cannot really get the "true" prediction for supertype level, since 20 % are missing for each cluster. Also, these are only the preds based on the Yao data, excluding zhang data.

Hence, to get the fairest comparison, ABC predictions are generated by taking Yao and Zhang cells restricted to our own data boundary and using them to calculate cell type proportions per brain region.


In [50]:
abc_df = pd.concat([yao.obs.loc[:,["allen_acronym", allen_level_to_analyze]], 
                    zhang.obs.loc[:,["allen_acronym", allen_level_to_analyze]]], axis=0)

abc_df = abc_df.reset_index(drop=True)

abc_df = abc_df.dropna()
abc_df.allen_acronym = abc_df.allen_acronym.replace(hmv1_like_acro_summarizer)

calculate fraction of cells per brain region

In [51]:
abc_pred = (
    abc_df
    .groupby(["allen_acronym", allen_level_to_analyze])
    .size()
    .unstack(fill_value=0)
)

abc_pred = abc_pred.div(abc_pred.sum(axis=0), axis=1)

In [52]:
# add back all nuclei that were not predicted

not_included_nucs = [n for n in hm3d_pred.index if n not in abc_pred.index]

zeros_df = pd.DataFrame(
    index = not_included_nucs,
    columns = abc_pred.columns,
    data = np.zeros(shape=(len(not_included_nucs), len(abc_pred.columns)))
)

abc_pred = pd.concat([abc_pred, zeros_df], axis=0)

#reorder index. this will drop NA "nucleus"
abc_pred = abc_pred.loc[hm3d_pred.index, :] 

In [53]:
abc_pred.head(2)

supertype,0001 CLA-EPd-CTX Car3 Glut_1,0002 CLA-EPd-CTX Car3 Glut_2,0003 IT EP-CLA Glut_1,0004 IT EP-CLA Glut_2,0005 IT EP-CLA Glut_3,0006 IT EP-CLA Glut_4,0007 L5/6 IT TPE-ENT Glut_1,0010 L5/6 IT TPE-ENT Glut_4,0023 L4/5 IT CTX Glut_1,0024 L4/5 IT CTX Glut_2,...,1192 SMC NN_1,1193 Endo NN_1,1194 Microglia NN_1,1195 BAM NN_1,1196 Monocytes NN_1,1197 DC NN_1,1198 B cells NN_1,1199 ILC NN_2,1200 NK cells NN_3,1201 T cells NN_4
allen_acronym,,,,,,,,,,,,,,,,,,,,,
AAA,0.0,0.0,0.0,0.0,0.0,0.0,0.392857,0.0,0.0,0.5,...,0.028466,0.010029,0.010345,0.010540,0.0,0.00000,0.005650,0.0,0.009709,0.017857
ACB,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.012285,0.020760,0.028973,0.012297,0.0,0.00885,0.022599,0.0,0.009709,0.011905


# Let's compare the predictions 

### Let's get the neuronal cell types, find their best matches, and create mappers

For the time being, all cell types and their matches will be obtained, regardless of how good the match was.

In [54]:
hm3d_to_abc_mapper = translator.loc[
    (translator.source_clustering == f"{hm3d_level_to_analyze}_descr") & 
    (~translator.source_label.str.contains("NN")) & 
    (translator.target_clustering == f"{allen_level_to_analyze}_id_label_propagated"), 
    :].set_index(f"source_label").to_dict()[f"most_probable_match"]

hm3d_to_hmv1_mapper = translator.loc[
    (translator.source_clustering == f"{hm3d_level_to_analyze}_descr") & 
    (~translator.source_label.str.contains("NN")) & 
    (translator.target_clustering == f"{hmv1_level_to_analyze}_propagated"), 
    :].set_index(f"source_label").to_dict()[f"most_probable_match"]

### Let's compare across all regions

but remove fiber tracts and HY from analysis by filtering those nuclei and then renormalizing

In [55]:
def remove_nucs_and_normalize(df):
    df = df[~df.index.to_series().apply(lambda x: x.islower())] #fiber tracts all lower case
    df = df[df.index.to_series() != "HY"] 
    df = df.div(df.sum(axis=0), axis=1)
    return df

hm3d_pred = remove_nucs_and_normalize(hm3d_pred)
hmv1_pred = remove_nucs_and_normalize(hmv1_pred)
abc_pred  = remove_nucs_and_normalize(abc_pred)


### Calculating continuous overlap percentages

In [56]:

def compute_overlap_dataframe(hm3d_pred, other_pred, mapper):
    records = []

    for cluster, matched_cluster in mapper.items():

        if pd.isna(matched_cluster):
            print(f"{matched_cluster=} not in mapper")
            records.append({
                'hm3d_cluster': cluster,
                'matched_cluster': np.nan,
                'overlap_percent': np.nan
            })
            continue

        if matched_cluster not in other_pred.columns or cluster not in hm3d_pred.columns:
            print(f"{matched_cluster=}: cluster pairs are not in other's columns")
            records.append({
                'hm3d_cluster': cluster,
                'matched_cluster': matched_cluster,
                'overlap_percent': np.nan
            })
            continue

        # Get predicted values for both clusters
        hm3d_vals = hm3d_pred[cluster]
        other_vals = other_pred[matched_cluster]

        # Compute overlap
        overlap_sum = (hm3d_vals.combine(other_vals, func=min)).sum()
        total_sum = hm3d_vals.sum()

        overlap_percent = overlap_sum / total_sum if total_sum > 0 else 0

        records.append({
            'hm3d_cluster': cluster,
            'matched_cluster': matched_cluster,
            'overlap_percent': overlap_percent
        })

    return pd.DataFrame(records)


In [57]:
overlap_vs_hmv1 = compute_overlap_dataframe(hm3d_pred, hmv1_pred, hm3d_to_hmv1_mapper)
overlap_vs_abc = compute_overlap_dataframe(hm3d_pred, abc_pred, hm3d_to_abc_mapper)

in hmv1 there can be perfect overlap if all values are nan becuase there was no prediction for the mapping cluster

In [58]:
overlap_vs_hmv1[overlap_vs_hmv1.overlap_percent == 1]

,hm3d_cluster,matched_cluster,overlap_percent
4,Gb3-128 SI Chat,C286-118: Tacr3.Onecut3.Onecut2.GABA-2,1.0
8,Gb3-132 CP Nfix,C286-151: St18.Lhx6.GABA-1,1.0
13,Gb3-137 LHA Tac2,C286-171: Tac2.Nts.GABA-1,1.0
24,Gb3-148 MEA Prox1,C286-128: Aldh3b2.GABA-2,1.0
25,Gb3-149 MEA Nxph2,C286-128: Aldh3b2.GABA-2,1.0
27,Gb3-151 BST Satb2,C286-100: Cxcl14.Pnoc.Mixed.GABA-2,1.0
29,Gb3-153 PS Gli3,C286-161: Lhx6.Lhx8.GABA-1,1.0
30,Gb3-154 MPO Cdh23,C286-161: Lhx6.Lhx8.GABA-1,1.0
32,Gb3-156 LPO Ntn1,C286-161: Lhx6.Lhx8.GABA-1,1.0
33,Gb3-157 MPO Zfp536,C286-161: Lhx6.Lhx8.GABA-1,1.0


In [59]:
weird_clusters = overlap_vs_hmv1[overlap_vs_hmv1.overlap_percent == 1].hm3d_cluster.values

for weird in weird_clusters:
    assert len(hmv1[hmv1.obs[f"{hmv1_level_to_analyze}"] == weird].obs.Region_predicted.unique()) == 0, "some of the weird clusters actually have predicted regions"

Hence, replace those values with "match not predicted in hmv1"

In [60]:
overlap_vs_hmv1.loc[overlap_vs_hmv1.overlap_percent == 1, "overlap_percent"] = "match not predicted in hmv1"

/tmp/ipykernel_3905278/3115152059.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'match not predicted in hmv1' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  overlap_vs_hmv1.loc[overlap_vs_hmv1.overlap_percent == 1, "overlap_percent"] = "match not predicted in hmv1"


In [61]:
not_predicted_clusters_hmv1 = overlap_vs_hmv1.loc[overlap_vs_hmv1.overlap_percent == "match not predicted in hmv1"].matched_cluster.to_list()

In [62]:
# join

overlaps = overlap_vs_abc.set_index("hm3d_cluster").join(
    overlap_vs_hmv1.set_index("hm3d_cluster"),
    how="left",
    lsuffix="_abc",
    rsuffix="_hmv1"
)


In [63]:
overlaps.head()

,matched_cluster_abc,overlap_percent_abc,matched_cluster_hmv1,overlap_percent_hmv1
hm3d_cluster,,,,
Gb3-124 MPO Meis1,0397 SI-MPO-LPO Lhx8 Gaba_2,0.663258,C286-165: Crh.Zeb2.Lhx8.GABA-1,0.144937
Gb3-125 SI Lgr5,0396 SI-MPO-LPO Lhx8 Gaba_1,0.626308,C286-163: Gbx1.Sox6.Lhx8.GABA-1,0.030166
Gb3-126 MPO Sema3d,0399 SI-MPO-LPO Lhx8 Gaba_4,0.394633,C286-165: Crh.Zeb2.Lhx8.GABA-1,0.067846
Gb3-127 MPO Penk,0398 SI-MPO-LPO Lhx8 Gaba_3,0.531857,C286-164: Nmu.Zeb2.Lhx8.GABA-1,0.030997
Gb3-128 SI Chat,0400 SI-MPO-LPO Lhx8 Gaba_5,0.359357,C286-118: Tacr3.Onecut3.Onecut2.GABA-2,match not predicted in hmv1


### Calculating overlap of top n brain regions 

In [64]:

def compute_top_n_nucleus_overlap_count_with_topn(hm3d_pred, other_pred, mapper, top_n=10):
    records = []

    for cluster, matched_cluster in mapper.items():

        # Always compute top N HM3D nuclei (non-zero predictions only)
        if cluster in hm3d_pred.columns:
            nonzero_hm3d = hm3d_pred[cluster][hm3d_pred[cluster] > 0]
            top_n_list_hm3d = list(nonzero_hm3d.nlargest(top_n).index)
        else:
            top_n_list_hm3d = np.nan

        # Check if valid mapping and both clusters exist
        if (
            pd.isna(matched_cluster)
            or matched_cluster not in other_pred.columns
            or cluster not in hm3d_pred.columns
        ):
            records.append({
                'hm3d_cluster': cluster,
                'matched_cluster': matched_cluster if not pd.isna(matched_cluster) else np.nan,
                f'top_{top_n}_nuclei_intersection': np.nan,
                f'top_{top_n}_nuclei_hm3d': top_n_list_hm3d,
                f'top_{top_n}_nuclei_other': np.nan
            })
            continue

        # Compute top N nuclei from the other dataset (non-zero only)
        nonzero_other = other_pred[matched_cluster][other_pred[matched_cluster] > 0]
        top_n_list_other = list(nonzero_other.nlargest(top_n).index)

        # Compute intersection
        intersection_count = len(set(top_n_list_hm3d).intersection(set(top_n_list_other)))

        records.append({
            'hm3d_cluster': cluster,
            'matched_cluster': matched_cluster,
            f'top_{top_n}_nuclei_intersection': intersection_count,
            f'top_{top_n}_nuclei_hm3d': top_n_list_hm3d,
            f'top_{top_n}_nuclei_other': top_n_list_other
        })

    return pd.DataFrame(records)


In [65]:
top_n = 3 

# Compare top-N nuclei overlap vs abc
top_n_overlap_abc = compute_top_n_nucleus_overlap_count_with_topn(
    hm3d_pred, abc_pred, hm3d_to_abc_mapper, top_n=top_n
)

# Compare top-N nuclei overlap vs hmv1
top_n_overlap_hmv1 = compute_top_n_nucleus_overlap_count_with_topn(
    hm3d_pred, hmv1_pred, hm3d_to_hmv1_mapper, top_n=top_n
)

In [66]:
# flag the clusters whose matches were not predicted in hmv1

top_n_overlap_hmv1.loc[top_n_overlap_hmv1.hm3d_cluster.isin(not_predicted_clusters_hmv1), f"top_{top_n}_nuclei_intersection"] = "match not predicted in hmv1"

/tmp/ipykernel_3905278/1262695459.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'match not predicted in hmv1' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  top_n_overlap_hmv1.loc[top_n_overlap_hmv1.hm3d_cluster.isin(not_predicted_clusters_hmv1), f"top_{top_n}_nuclei_intersection"] = "match not predicted in hmv1"


In [67]:
overlaps_top_n = top_n_overlap_abc.set_index("hm3d_cluster").join(
    top_n_overlap_hmv1.set_index("hm3d_cluster"),
    how="left",
    lsuffix="_abc",
    rsuffix="_hmv1"
)

In [68]:
all_overlaps = overlaps.loc[:, [c for c in overlaps.columns if "matched" not in c]].join(
    overlaps_top_n.loc[:, [c for c in overlaps_top_n.columns if "hm3d" not in c]],
    how="left",
)
all_overlaps[f"top_{top_n}_nuclei_hm3d"] = overlaps_top_n[f"top_{top_n}_nuclei_hm3d_abc"].values

In [69]:
all_overlaps.head(2)

,overlap_percent_abc,overlap_percent_hmv1,matched_cluster_abc,top_3_nuclei_intersection_abc,top_3_nuclei_other_abc,matched_cluster_hmv1,top_3_nuclei_intersection_hmv1,top_3_nuclei_other_hmv1,top_3_nuclei_hm3d
hm3d_cluster,,,,,,,,,
Gb3-124 MPO Meis1,0.663258,0.144937,0397 SI-MPO-LPO Lhx8 Gaba_2,2,"[MPO, LPO, ADP]",C286-165: Crh.Zeb2.Lhx8.GABA-1,1,[MPO],"[MPO, LPO, BST]"
Gb3-125 SI Lgr5,0.626308,0.030166,0396 SI-MPO-LPO Lhx8 Gaba_1,2,"[SI, BST, GPe]",C286-163: Gbx1.Sox6.Lhx8.GABA-1,0,[LPO],"[SI, GPe, NDB]"


next, obtain categories: 

full overlap, partial overlap, no match.

for abc, this works out of the box.

for hmv1, we will only compare as many nuclei as there were nuclei predicted by hmv1

In [70]:

def compute_match_category(df, hm3d_col, other_col):
    # Extract the suffix from the other_col (e.g., 'hmv1' from 'top_3_nuclei_other_hmv1')
    suffix = other_col.split('_')[-1]
    new_col_name = f"hm3d_to_{suffix}_match_category"

    def categorize_match(row):
        other_val = row[other_col]
        hm3d_val = row[hm3d_col]

        if other_val == "no match":
            return "no cluster match"
        elif isinstance(other_val, list) and len(other_val) == 0:
            return "matching cluster has no prediction"
        elif isinstance(other_val, list) and isinstance(hm3d_val, list):
            # Trim hm3d_val to same length as other_val
            trimmed_hm3d = hm3d_val[:len(other_val)]
            intersection = set(trimmed_hm3d).intersection(other_val)
            if len(intersection) == 0:
                return "no overlap"
            elif len(intersection) < len(other_val):
                return "partial overlap"
            else:
                return "full overlap"
        else:
            return "invalid format"

    df[new_col_name] = df.apply(categorize_match, axis=1)
    return df


In [71]:
all_overlaps = compute_match_category(df = all_overlaps, hm3d_col = "top_3_nuclei_hm3d", other_col = "top_3_nuclei_other_hmv1")
all_overlaps = compute_match_category(df = all_overlaps, hm3d_col = "top_3_nuclei_hm3d", other_col = "top_3_nuclei_other_abc")

In [72]:
all_overlaps = all_overlaps.loc[:, [

    "top_3_nuclei_hm3d",

    "matched_cluster_abc",
    "overlap_percent_abc",
    "top_3_nuclei_other_abc",
    "top_3_nuclei_intersection_abc",
    "hm3d_to_abc_match_category",

    "matched_cluster_hmv1",
    "overlap_percent_hmv1",
    "top_3_nuclei_other_hmv1",
    "top_3_nuclei_intersection_hmv1",
    "hm3d_to_hmv1_match_category",

]]

In [73]:
all_overlaps

,top_3_nuclei_hm3d,matched_cluster_abc,overlap_percent_abc,top_3_nuclei_other_abc,top_3_nuclei_intersection_abc,hm3d_to_abc_match_category,matched_cluster_hmv1,overlap_percent_hmv1,top_3_nuclei_other_hmv1,top_3_nuclei_intersection_hmv1,hm3d_to_hmv1_match_category
hm3d_cluster,,,,,,,,,,,
Gb3-124 MPO Meis1,"[MPO, LPO, BST]",0397 SI-MPO-LPO Lhx8 Gaba_2,0.663258,"[MPO, LPO, ADP]",2,partial overlap,C286-165: Crh.Zeb2.Lhx8.GABA-1,0.144937,[MPO],1,full overlap
Gb3-125 SI Lgr5,"[SI, GPe, NDB]",0396 SI-MPO-LPO Lhx8 Gaba_1,0.626308,"[SI, BST, GPe]",2,partial overlap,C286-163: Gbx1.Sox6.Lhx8.GABA-1,0.030166,[LPO],0,no overlap
Gb3-126 MPO Sema3d,"[SI, LSr, MPO]",0399 SI-MPO-LPO Lhx8 Gaba_4,0.394633,"[MPO, ADP, BST]",1,partial overlap,C286-165: Crh.Zeb2.Lhx8.GABA-1,0.067846,[MPO],1,no overlap
Gb3-127 MPO Penk,"[MPO, MPN, SI]",0398 SI-MPO-LPO Lhx8 Gaba_3,0.531857,"[MPO, MPN, AVPV]",2,partial overlap,C286-164: Nmu.Zeb2.Lhx8.GABA-1,0.030997,[LPO],0,no overlap
Gb3-128 SI Chat,"[SI, CP, GPe]",0400 SI-MPO-LPO Lhx8 Gaba_5,0.359357,"[SI, LPO, MPO]",1,partial overlap,C286-118: Tacr3.Onecut3.Onecut2.GABA-2,match not predicted in hmv1,[],0,matching cluster has no prediction
...,...,...,...,...,...,...,...,...,...,...,...
Gl3-119 AVPV Agtr1a,"[AVPV, MS, MEPO]",0521 AVPV-MEPO-SFO Tbr1 Glut_3,0.408703,"[MEPO, AVPV, OV]",2,partial overlap,C286-21: Gng8.Samd3.GLU-2,0.023535,[LHA],0,no overlap
Gl3-120 MEPO Nwd2,"[MEPO, V3, AVPV]",0521 AVPV-MEPO-SFO Tbr1 Glut_3,0.416360,"[MEPO, AVPV, OV]",2,partial overlap,C286-21: Gng8.Samd3.GLU-2,0.005852,[LHA],0,no overlap
Gl3-121 AVPV Gpc6,"[AVPV, MPO, LHA]",0520 AVPV-MEPO-SFO Tbr1 Glut_2,0.497764,"[AVPV, MPO, MEPO]",2,partial overlap,C286-21: Gng8.Samd3.GLU-2,0.068782,[LHA],1,no overlap


In [74]:
all_overlaps.to_csv(OUTDIR / f"region_prediction_comparison_to_other_atlases_{hm3d_level_to_analyze}.csv")